# 01 — Data Exploration

Before running any statistical tests we need to understand the shape, coverage, and
quality of every CSV exported from Prometheus.
This notebook audits all data files and flags gaps or outliers that must be addressed
before the latency / throughput analysis notebooks run.

## Imports and plot style

We import shared utilities from `scripts/utils.py` and configure matplotlib
for publication-quality output (serif font, no top/right spines, light grid).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'scripts'))
from utils import (
    DATA_DIR, SERVICES, SERVICE_LABELS, SERVICE_COLORS,
    load_csv, describe_series, set_plot_style,
)

set_plot_style()
%matplotlib inline
print('Data directory:', DATA_DIR)

## Load all CSV files

We attempt to load every expected CSV.  Missing files are recorded in `missing`
so downstream cells can skip them gracefully — this prevents cascading errors
when only a subset of benchmark scenarios has been run.

In [ ]:
METRICS = [
    'latency_p50_dotnet', 'latency_p50_go',
    'latency_p75_dotnet', 'latency_p75_go',
    'latency_p90_dotnet', 'latency_p90_go',
    'latency_p95_dotnet', 'latency_p95_go',
    'latency_p99_dotnet', 'latency_p99_go',
    'throughput_dotnet',  'throughput_go',
    'error_rate_dotnet',  'error_rate_go',
    'cpu_dotnet',         'cpu_go',
    'memory_heap_dotnet', 'memory_heap_go',
    'consumer_lag_dotnet','consumer_lag_go',
]

frames = {}
missing = []
for name in METRICS:
    try:
        frames[name] = load_csv(f'{name}.csv')
    except FileNotFoundError:
        missing.append(name)

print(f'Loaded:  {len(frames)} / {len(METRICS)} files')
if missing:
    print(f'Missing: {missing}')

for name, df in frames.items():
    print(f'  {name:30s}  shape={str(df.shape):12s}  dtypes={df.dtypes.to_dict()}')

## Missing value check

Prometheus may return gaps (e.g. during a scrape failure or service restart).
We print the NaN count per column for every loaded frame.

In [ ]:
print(f'{'File':<32} {'Column':<15} {'NaN count':>10} {'NaN %':>8}')
print('-' * 68)
for name, df in frames.items():
    for col in df.columns:
        n_nan = int(df[col].isna().sum())
        pct   = 100 * n_nan / max(len(df), 1)
        if n_nan > 0:
            print(f'{name:<32} {col:<15} {n_nan:>10} {pct:>7.1f}%')

print('\n(No output above = no missing values found)')

## Time range coverage

All latency and throughput CSVs should share the same time window — the k6
benchmark run.  Mismatched ranges suggest a partial export or a clock skew.

In [ ]:
print(f'{'File':<32} {'Start (UTC)':<28} {'End (UTC)':<28} {'Points':>7}')
print('-' * 100)
for name, df in frames.items():
    if df.index.dtype == 'datetime64[ns, UTC]' or hasattr(df.index, 'tz'):
        start = str(df.index.min())
        end   = str(df.index.max())
    else:
        start = end = 'non-datetime index'
    print(f'{name:<32} {start:<28} {end:<28} {len(df):>7}')

## Descriptive statistics — latency series

A quick `describe()` on every latency CSV gives an at-a-glance view of the
distribution before formal testing.  P99 >> P95 is expected; extreme max values
suggest individual timeout events.

In [ ]:
latency_frames = {k: v for k, v in frames.items() if k.startswith('latency_p95')}
for name, df in latency_frames.items():
    col = df.columns[0]
    print(f'\n=== {name} ({col}) ===')
    print(df[col].describe(percentiles=[0.5, 0.75, 0.95, 0.99]).to_string())

## Raw latency time series

Plotting the raw P95 series for both services side by side reveals the
ramp-up phase, sustained load plateau, and ramp-down — and whether the two
services track each other or diverge under load.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for svc in SERVICES:
    key = f'latency_p95_{svc}'
    if key in frames:
        df = frames[key]
        col = df.columns[0]
        ax.plot(df.index, df[col], label=SERVICE_LABELS[svc], color=SERVICE_COLORS[svc])

ax.set_xlabel('Time (UTC)')
ax.set_ylabel('P95 Latency (ms)')
ax.set_title('Raw P95 Latency Time Series')
ax.legend()
plt.tight_layout()
plt.show()

## Data quality: gaps and outliers

Outliers beyond 3σ likely correspond to timeout retries or JVM GC pauses.
We report them but do **not** remove them — they represent real system behaviour
and will be visible in the CDF (notebook 02).

In [ ]:
issues = []
for name, df in frames.items():
    for col in df.select_dtypes(include='number').columns:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        mean, std = s.mean(), s.std()
        outliers  = s[s > mean + 3 * std]
        # Gap: consecutive timestamp difference > 30 s
        if hasattr(df.index, 'to_series'):
            diffs = df.index.to_series().diff().dt.total_seconds()
            gaps  = int((diffs > 30).sum())
        else:
            gaps = 0
        if len(outliers) > 0 or gaps > 0:
            issues.append({'file': name, 'column': col,
                           'outliers_3sigma': len(outliers),
                           'timestamp_gaps':  gaps})

if issues:
    print(pd.DataFrame(issues).to_string(index=False))
else:
    print('No outliers >3sigma and no timestamp gaps detected.')

## Conclusion

Update this cell after reviewing the output above.

**Template**:
- All `N` data files loaded successfully.
- Time ranges are aligned: `HH:MM–HH:MM UTC`.
- Missing values: none / X NaN rows in `Y` (acceptable — scrape gaps during startup).
- Outliers: X points beyond 3σ in P99 latency — retained as genuine tail events.
- Data is **ready** for statistical analysis (notebooks 02–06).